# Copyright

<PRE>
Jelen iPython notebook a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott "Deep Learning a gyakorlatban Python és LUA alapon" tantárgy segédanyagaként készült.
A tantárgy honlapja: http://smartlab.tmit.bme.hu/oktatas-deep-learning
Deep Learning kutatás: http://smartlab.tmit.bme.hu/deep-learning
Jelen notebook Nicolas P. Rougier munkája alapján készült, melyet BSD licensz véd: http://www.labri.fr/perso/nrougier/downloads/mlp.py

A notebook bármely részének újra felhasználása, publikálása csak a szerzők írásos beleegyezése esetén megegengedett.

2026 (c) Gyires-Tóth Bálint (toth.b kukac tmit pont bme pont hu), Zainkó Csaba
</PRE>

# 1. feladat – Transfer learning konvolúciós neurális hálóval

**Képosztályozás kevés adatból: virágfajták felismerése előtanított MobileNetV2 hálóval (Keras)**

Ebben a feladatban **nem kell kódot írnod**: a notebook kész, cellánként végig kell futtatnod. Közben olvasd el figyelmesen a magyarázatokat, és nézd meg az ábrákat. A notebook végén néhány kérdésre kell röviden válaszolnod, ez a beadandó.

### Mit fogsz megtanulni?
- hogyan épül fel egy konvolúciós háló (CNN), és mit „látnak” a különböző mélységű rétegei,
- mi a **transfer learning**, és miért működik,
- mi a különbség a **feature extraction** és a **fine-tuning** között,
- hogyan értékelünk ki egy osztályozó modellt (tanulási görbék, konfúziós mátrix, hibás példák).

### Használat
1. **Kapcsold be a GPU-t:** *Runtime → Change runtime type → T4 GPU* (magyar felületen: *Futtatókörnyezet → Futtatókörnyezet típusának módosítása*).
2. A cellákat sorban futtasd: kattints a cellára, majd nyomd meg a **Shift + Enter** billentyűket. Az egészet egyben is futtathatod: *Runtime → Run all*.
3. A teljes futás GPU-n kb. **10–15 perc**.

> 💡 Ha futás közben megszakad a kapcsolat vagy hibát kapsz, futtasd újra a notebookot az elejétől (*Runtime → Restart session and run all*).

## 0. Egy kis elmélet

### Konvolúció
A konvolúciós réteg egy kis méretű (pl. 3×3-as) **szűrőt** (*kernel*) csúsztat végig a képen. Minden pozícióban összeszorozza a szűrő elemeit az alatta lévő képrészlet pixeleivel, és ezeket összeadja. Az eredmény egy **feature map** (jellemzőtérkép), amely megmutatja, hol található a képen a szűrő által keresett minta (pl. egy függőleges él). A szűrők értékeit nem mi adjuk meg kézzel: **a háló tanulja meg őket**.

### A rétegek hierarchiája
Egy mély CNN-ben
- a **korai rétegek** egyszerű, általános mintákat ismernek fel: éleket, színátmeneteket, textúrákat;
- a **középső rétegek** ezekből összetettebb alakzatokat raknak össze: íveket, mintázatokat, formákat;
- a **késői rétegek** már feladatspecifikus, „objektumszintű” jellemzőket ismernek fel (pl. szirom, szem, kerék).

### Transfer learning
Egy nagy háló betanítása rengeteg adatot és számítási kapacitást igényel. Az **ImageNet** adathalmaz kb. 1,2 millió képet tartalmaz 1000 osztályban. Az ezen tanított hálók korai és középső rétegei olyan általános képi jellemzőket tanultak meg, amelyek **más feladatokban is hasznosak**. A transfer learning lényege, hogy egy ilyen **előtanított** háló tudását átvisszük a saját, jellemzően sokkal kisebb adathalmazunkra.

Két fő módja van:

| Módszer | Mit tanítunk? | Mikor jó? |
|---|---|---|
| **Feature extraction** | Az előtanított háló (a „törzs”, angolul *backbone*) súlyait **befagyasztjuk**, és csak egy új, kis osztályozó réteget tanítunk a tetején. | Kevés adat, gyors tanítás. |
| **Fine-tuning** | A feature extraction után a törzs **utolsó rétegeit is felengedjük**, és nagyon kicsi tanulási rátával finomhangoljuk őket. | Ha még néhány százalékot szeretnénk javítani. |

Ebben a notebookban mindkettőt kipróbáljuk, és összehasonlítjuk őket egy **nulláról tanított** kis CNN-nel.

## 1. A környezet előkészítése

Betöltjük a szükséges könyvtárakat, és ellenőrizzük, hogy elérhető-e a GPU.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("TensorFlow verzió:", tf.__version__)
print("Keras verzió:     ", keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("✅ GPU elérhető:", gpus[0].name)
else:
    print("⚠️  Nincs GPU! A notebook így is lefut, de sokkal lassabban.")
    print("   Kapcsold be: Runtime → Change runtime type → T4 GPU, majd futtasd újra a cellát.")

### Beállítások

Ezek a notebook fő paraméterei. **Első futtatáskor ne változtass rajtuk!** A notebook végén, a bónuszkérdésnél kísérletezhetsz velük.

A `SEED` (véletlenszám-mag) rögzítése miatt az eredmények nagyjából reprodukálhatók, de GPU-n kis eltérések előfordulhatnak.

In [ ]:
SEED = 42              # véletlenszám-mag
IMG_SIZE = 160         # a képek ekkora (IMG_SIZE × IMG_SIZE) méretre lesznek átméretezve
BATCH_SIZE = 32        # ennyi képet dolgoz fel a háló egyszerre

EPOCHS_BASELINE = 15   # epochok száma a nulláról tanított CNN-hez
EPOCHS_FEATURE = 10    # epochok száma a feature extraction lépéshez
EPOCHS_FINETUNE = 10   # epochok száma a fine-tuning lépéshez

LR_FEATURE = 1e-3      # tanulási ráta a feature extraction lépéshez
LR_FINETUNE = 1e-5     # tanulási ráta a fine-tuninghoz (100-szor kisebb!)
FINE_TUNE_AT = 100     # a MobileNetV2 ennyiedik rétegétől kezdve engedjük fel a súlyokat

TRAIN_SPLIT = "train[:20%]"     # tanítóhalmaz (az adatok 20%-a)
VAL_SPLIT = "train[20%:30%]"    # validációs halmaz (10%)
TEST_SPLIT = "train[30%:50%]"   # teszthalmaz (20%)

keras.utils.set_random_seed(SEED)

## 2. Az adathalmaz

A **tf_flowers** adathalmazt használjuk: kb. 3670 fénykép 5 virágfajtáról (pitypang, százszorszép, tulipán, napraforgó, rózsa).

A transfer learning ereje akkor látszik igazán, ha **kevés a tanítóadat**, ezért szándékosan csak az adatok 20%-ával tanítunk (kb. 730 kép, osztályonként nagyjából 150).

| Részhalmaz | Arány | Szerep |
|---|---|---|
| tanító (*train*) | 20% | ezen tanul a modell |
| validációs (*validation*) | 10% | tanítás közben ezzel figyeljük, mennyire általánosít a modell |
| teszt (*test*) | 20% | csak a legvégén használjuk, a végső kiértékeléshez |

> Miért kell külön validációs és teszthalmaz? A validációs eredményeket figyelve döntéseket hozunk (pl. hány epochig tanítsunk), így közvetve „ráhangolódunk”. A teszthalmaz ettől független, ezért ad őszinte képet a modell teljesítményéről.

Az első letöltés kb. 1 percig tart.

In [ ]:
!pip install -q importlib_resources

In [ ]:
(train_raw, val_raw, test_raw), info = tfds.load(
    "tf_flowers",
    split=[TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT],
    as_supervised=True,
    with_info=True,
)

EN_NAMES = info.features["label"].names
HU = {"dandelion": "pitypang", "daisy": "százszorszép", "tulips": "tulipán",
      "sunflowers": "napraforgó", "roses": "rózsa"}
CLASS_NAMES = [HU.get(name, name) for name in EN_NAMES]
NUM_CLASSES = len(CLASS_NAMES)

print("Osztályok:", CLASS_NAMES)
print(f"Tanítóképek:    {int(train_raw.cardinality())}")
print(f"Validációs képek: {int(val_raw.cardinality())}")
print(f"Tesztképek:     {int(test_raw.cardinality())}")

Nézzük meg, hány kép jut egy-egy osztályra a tanítóhalmazban. Ha nagyon kiegyensúlyozatlan lenne az eloszlás, a modell a gyakori osztályok felé torzulhatna.

In [ ]:
train_labels = np.array([int(label) for _, label in train_raw])
counts = np.bincount(train_labels, minlength=NUM_CLASSES)

plt.figure(figsize=(7, 3.5))
plt.bar(CLASS_NAMES, counts, color="tab:green")
for i, c in enumerate(counts):
    plt.text(i, c + 2, str(c), ha="center")
plt.title("Képek száma osztályonként (tanítóhalmaz)")
plt.ylabel("képek száma")
plt.ylim(0, counts.max() * 1.15)
plt.show()

Néhány mintakép a tanítóhalmazból. Figyeld meg, hogy a képek **különböző méretűek**, és a virágok sokféle háttér előtt, különböző nézőpontból szerepelnek.

In [ ]:
plt.figure(figsize=(12, 7))
for i, (image, label) in enumerate(train_raw.take(15)):
    plt.subplot(3, 5, i + 1)
    plt.imshow(image.numpy())
    plt.title(f"{CLASS_NAMES[int(label)]}\n{image.shape[1]}×{image.shape[0]} px", fontsize=9)
    plt.axis("off")
plt.suptitle("Mintaképek a tanítóhalmazból (eredeti méretben)")
plt.tight_layout()
plt.show()

## 3. Előfeldolgozás

- Minden képet **160×160 pixelesre** méretezünk, mert a háló fix méretű bemenetet vár.
- A képeket **batchekbe** (kötegekbe) rendezzük: egyszerre 32 kép megy át a hálón.
- A `cache()` a memóriában tartja a már feldolgozott képeket, a `prefetch()` pedig előre betölti a következő batchet, amíg a GPU az előzőn dolgozik. Mindkettő csak gyorsít.

A pixelértékek skálázását (normalizálását) **a modellen belül** végezzük, mert a két modellünk eltérő skálát vár (a nulláról tanított CNN [0, 1] tartományt, a MobileNetV2 [−1, 1] tartományt).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def resize(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))   # float32, 0–255 értékek
    return image, label


train_ds = (train_raw.map(resize, num_parallel_calls=AUTOTUNE)
            .cache()
            .shuffle(1000, seed=SEED)       # tanításnál minden epochban más sorrend
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = val_raw.map(resize, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(resize, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)

for images, labels in train_ds.take(1):
    print("Egy batch képeinek alakja:", images.shape)   # (batch, magasság, szélesség, színcsatornák)
    print("Egy batch címkéinek alakja:", labels.shape)

### Adatbővítés (*data augmentation*)

Kevés adatnál a modell hajlamos „bemagolni” a tanítóképeket ahelyett, hogy általános szabályokat tanulna. Ezt **túltanulásnak** (*overfitting*) nevezzük: a tanítóhalmazon nagyon jó a pontosság, új képeken viszont gyenge.

Ezért tanítás közben minden képet véletlenszerűen kicsit módosítunk: tükrözzük, elforgatjuk, nagyítjuk. Egy tükrözött tulipán is tulipán marad, a modell viszont így minden epochban kicsit „új” képeket lát.

Az adatbővítő rétegek **csak tanításkor aktívak**, kiértékeléskor a Keras automatikusan kikapcsolja őket.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),    # ±10% (kb. ±36°) forgatás
    layers.RandomZoom(0.1),        # ±10% nagyítás
], name="adatbovites")

sample_image = images[0]
sample_label = int(labels[0])

plt.figure(figsize=(8, 8))
for i in range(9):
    augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(np.clip(np.array(augmented[0]) / 255.0, 0, 1))
    plt.axis("off")
plt.suptitle(f"Ugyanaz a kép ({CLASS_NAMES[sample_label]}) 9 véletlen módosítással")
plt.tight_layout()
plt.show()

### Segédfüggvények

Két kis függvény, amelyeket később többször használunk: az egyik a tanulási görbéket rajzolja ki, a másik megszámolja a modell tanítható és befagyasztott paramétereit. Az eredményeket a `results` táblázatban gyűjtjük.

In [ ]:
results = {}   # modell neve -> (teszt pontosság, tanítható paraméterek száma)


def plot_history(history, title):
    h = history.history
    epochs = [e + 1 for e in history.epoch]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, h["accuracy"], "o-", label="tanító")
    ax1.plot(epochs, h["val_accuracy"], "o-", label="validációs")
    ax1.set_title("Pontosság (accuracy)")
    ax1.set_xlabel("epoch")
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax2.plot(epochs, h["loss"], "o-", label="tanító")
    ax2.plot(epochs, h["val_loss"], "o-", label="validációs")
    ax2.set_title("Veszteség (loss)")
    ax2.set_xlabel("epoch")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def count_params(model):
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    frozen = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    total = trainable + frozen
    print(f"Tanítható paraméterek:     {trainable:>10,}  ({100 * trainable / total:.1f}%)")
    print(f"Nem tanítható paraméterek: {frozen:>10,}")
    print(f"Összesen:                  {total:>10,}")
    return trainable

## 4. Viszonyítási alap: kis CNN nulláról tanítva

Először egy egyszerű konvolúciós hálót tanítunk **véletlen kezdősúlyokkal**, csak a saját ~730 képünkből. Ez lesz a *baseline*, vagyis a viszonyítási alap.

Felépítése:

| Réteg | Mit csinál? |
|---|---|
| adatbővítés | véletlen tükrözés, forgatás, nagyítás (csak tanításkor) |
| `Rescaling(1/255)` | a pixelértékeket [0, 1] közé skálázza |
| 3 × (`Conv2D` + `MaxPooling2D`) | 32, 64, majd 128 darab 3×3-as szűrő; a pooling felére csökkenti a térbeli méretet |
| `GlobalAveragePooling2D` | minden feature mapet egyetlen számmá átlagol |
| `Dropout(0.3)` | tanításkor véletlenszerűen kinullázza a neuronok 30%-át (a túltanulás ellen) |
| `Dense(5, softmax)` | 5 kimenet: az egyes osztályok valószínűségei |

**Előzetes tipp:** mit gondolsz, mennyire lesz jó egy háló, amely mindössze ~730 képből tanulja meg „látni” a virágokat?

In [ ]:
def build_baseline():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")


baseline = build_baseline()
baseline.summary()
baseline_trainable = count_params(baseline)

Tanítás. A kimenetben minden epochnál látod a tanító (`accuracy`, `loss`) és a validációs (`val_accuracy`, `val_loss`) eredményeket.

In [ ]:
baseline.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",   # a címkék egész számok (0–4)
    metrics=["accuracy"],
)

history_base = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE)

In [ ]:
plot_history(history_base, "Baseline CNN (nulláról tanítva)")

_, base_test_acc = baseline.evaluate(test_ds, verbose=0)
results["Baseline CNN (nulláról)"] = (base_test_acc, baseline_trainable)
print(f"Teszt pontosság: {base_test_acc:.1%}   (véletlen tippeléssel: {1 / NUM_CLASSES:.0%})")

## 5. Transfer learning, 1. lépés: feature extraction

Most a **MobileNetV2** hálót használjuk, amelyet a Google mobiltelefonokra tervezett: kicsi (kb. 2,3 millió paraméter) és gyors, ennek ellenére jó pontosságú. A súlyait az ImageNet adathalmazon tanították.

- `include_top=False`: az eredeti, 1000 ImageNet-osztályra tanított utolsó réteget **elhagyjuk**, mert nekünk csak 5 osztályunk van. Helyette sajátot teszünk a háló tetejére.
- `base_model.trainable = False`: a törzs összes súlyát **befagyasztjuk**, így tanításkor nem változnak.

A teljes modell felépítése:

`bemenet → adatbővítés → skálázás [−1, 1] közé → MobileNetV2 (befagyasztva) → GlobalAveragePooling → Dropout → Dense(5, softmax)`

> A `base_model(x, training=False)` hívás miatt a törzsben lévő **BatchNormalization** rétegek akkor is az ImageNeten tanult statisztikákat használják, amikor később a fine-tuningnál felengedjük őket. Enélkül a kis adathalmazunk statisztikái „elrontanák” az előtanított súlyokat.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

print("A MobileNetV2 rétegeinek száma:", len(base_model.layers))
print("A törzs kimenetének alakja:   ", base_model.output.shape)

A törzs egy 160×160×3-as képből **5×5×1280**-as jellemzőtömböt készít: 1280 feature mapet, mindegyik 5×5-ös. Ezt átlagoljuk (GlobalAveragePooling), így minden képből egy 1280 elemű **jellemzővektor** lesz. Erre tanítjuk az osztályozó réteget.

In [ ]:
rescale = layers.Rescaling(1.0 / 127.5, offset=-1, name="skalazas")   # 0–255 -> [-1, 1]
gap = layers.GlobalAveragePooling2D(name="atlagolas")
dropout = layers.Dropout(0.2, name="dropout")
classifier = layers.Dense(NUM_CLASSES, activation="softmax", name="osztalyozo")

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = rescale(x)
x = base_model(x, training=False)
x = gap(x)
x = dropout(x)
outputs = classifier(x)
tl_model = keras.Model(inputs, outputs, name="transfer_mobilenetv2")

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FEATURE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

tl_model.summary()
fe_trainable = count_params(tl_model)

Figyeld meg: a modell paramétereinek csak **töredéke** tanítható, a többi az előtanított, befagyasztott törzsben van.

Tanítás előtt az osztályozó réteg súlyai véletlenek, ezért a modell kb. véletlenszerűen tippel. Nézzük meg:

In [ ]:
_, acc_before = tl_model.evaluate(val_ds, verbose=0)
print(f"Validációs pontosság tanítás előtt: {acc_before:.1%}")

In [ ]:
history_fe = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FEATURE)

In [ ]:
plot_history(history_fe, "Transfer learning: feature extraction")

_, fe_test_acc = tl_model.evaluate(test_ds, verbose=0)
results["Feature extraction"] = (fe_test_acc, fe_trainable)
print(f"Teszt pontosság: {fe_test_acc:.1%}")

Érdekesség: lehet, hogy a validációs pontosság **magasabb**, mint a tanító. Ez nem hiba: tanításkor az adatbővítés és a Dropout „nehezíti” a feladatot, kiértékeléskor viszont mindkettő ki van kapcsolva.

## 6. Transfer learning, 2. lépés: fine-tuning

Most a törzs **felső rétegeit is felengedjük** (a 100. rétegtől a 154.-ig). Ezek a rétegek tanulták a legfeladatspecifikusabb, ImageNet-osztályokra hangolt jellemzőket, ezért ezeket érdemes a virágokhoz igazítani. A korai rétegek (élek, textúrák) általánosak, ezért azok befagyasztva maradnak.

Két fontos szabály:
1. **Csak feature extraction után** csináljuk. Ha a véletlen kezdősúlyú osztályozóval együtt rögtön a törzset is tanítanánk, a kezdeti nagy hibák „szétrombolnák” az előtanított súlyokat.
2. **Nagyon kicsi tanulási rátát** használunk (`1e-5`, ami 100-szor kisebb az eddiginél), hogy a súlyok csak finoman módosuljanak.

A módosítás után a modellt újra kell fordítani (`compile`), hogy a változás érvénybe lépjen.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

print(f"Felengedett rétegek a törzsben: {sum(l.trainable for l in base_model.layers)} / {len(base_model.layers)}")

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
ft_trainable = count_params(tl_model)

A tanítást ott folytatjuk, ahol a feature extraction abbahagyta (`initial_epoch`).

In [ ]:
history_ft = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FEATURE + EPOCHS_FINETUNE,
    initial_epoch=len(history_fe.epoch),
)

In [ ]:
plot_history(history_ft, "Transfer learning: fine-tuning")

_, ft_test_acc = tl_model.evaluate(test_ds, verbose=0)
results["Fine-tuning"] = (ft_test_acc, ft_trainable)
print(f"Teszt pontosság: {ft_test_acc:.1%}")

## 7. Összehasonlítás

Egy ábrán a három megközelítés validációs pontossága, alatta a teszthalmazon mért végeredmények.

In [ ]:
base_val = history_base.history["val_accuracy"]
tl_val = history_fe.history["val_accuracy"] + history_ft.history["val_accuracy"]
n_fe = len(history_fe.history["val_accuracy"])

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(base_val) + 1), base_val, "o-", label="Baseline CNN (nulláról)")
plt.plot(range(1, len(tl_val) + 1), tl_val, "o-", label="MobileNetV2 transfer learning")
plt.axvline(n_fe + 0.5, color="gray", ls="--")
plt.text(n_fe + 0.7, min(base_val), "fine-tuning\nkezdete", color="gray")
plt.axhline(1 / NUM_CLASSES, color="red", ls=":", label="véletlen tippelés")
plt.xlabel("epoch")
plt.ylabel("validációs pontosság")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.title("Validációs pontosság epochonként")
plt.show()

summary = pd.DataFrame(
    [(name, f"{acc:.1%}", f"{params:,}") for name, (acc, params) in results.items()],
    columns=["Modell", "Teszt pontosság", "Tanítható paraméterek"],
)
summary

## 8. A legjobb modell részletes kiértékelése

### Konfúziós mátrix

A pontosság egyetlen szám, nem mutatja meg, **milyen hibákat** követ el a modell. A konfúziós mátrix *i*-edik sorának *j*-edik eleme azt mutatja, hány *i* osztályú képet sorolt a modell a *j* osztályba. A főátlóban vannak a helyes, azon kívül a hibás döntések.

A `classification_report` osztályonként további mérőszámokat ad:
- **precision**: azok közül, amiket a modell X-nek mondott, hány volt valóban X,
- **recall**: a valódi X-ek közül hányat talált meg a modell,
- **f1-score**: a kettő harmonikus közepe.

In [ ]:
test_images = np.concatenate([x.numpy() for x, _ in test_ds]).astype("uint8")
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = tl_model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Greens", colorbar=False, xticks_rotation=45)
ax.set_xlabel("Becsült osztály")
ax.set_ylabel("Valós osztály")
ax.set_title("Konfúziós mátrix (teszthalmaz)")
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

### Hibásan osztályozott képek

Érdemes megnézni, hol téved a modell. Gyakran kiderül, hogy a hiba érthető: rossz a kép, több virág is van rajta, vagy a címke maga is vitatható.

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"Hibásan osztályozott tesztképek: {len(wrong)} / {len(y_true)}")

n_show = min(12, len(wrong))
plt.figure(figsize=(12, 9))
for i, idx in enumerate(wrong[:n_show]):
    plt.subplot(3, 4, i + 1)
    plt.imshow(test_images[idx])
    plt.title(f"valós: {CLASS_NAMES[y_true[idx]]}\n"
              f"tipp: {CLASS_NAMES[y_pred[idx]]} ({y_prob[idx].max():.0%})",
              fontsize=9, color="darkred")
    plt.axis("off")
plt.suptitle("Hibásan osztályozott tesztképek")
plt.tight_layout()
plt.show()

## 9. Mit „lát” a háló?

### 9.1 Az első konvolúciós réteg szűrői

A MobileNetV2 első rétege 32 darab 3×3×3-as szűrőt tartalmaz (3×3 pixel, 3 színcsatorna). Az alábbi ábrán ezeket színes képként rajzoltuk ki (felnagyítva). Ezek a szűrők a fine-tuning során befagyasztva maradtak, tehát **az ImageNeten tanult súlyokat** látod.

Figyeld meg, hogy sok szűrő egy-egy irányú **világos–sötét átmenetet** (élt) vagy egy **színt** keres.

In [ ]:
filters = base_model.get_layer("Conv1").get_weights()[0]
print("Az első konvolúciós réteg súlyainak alakja:", filters.shape, "(magasság, szélesség, bemeneti csatornák, szűrők)")

plt.figure(figsize=(10, 5))
for i in range(filters.shape[-1]):
    f = filters[:, :, :, i]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)   # 0–1 közé skálázás a megjelenítéshez
    plt.subplot(4, 8, i + 1)
    plt.imshow(f, interpolation="nearest")
    plt.axis("off")
plt.suptitle("A MobileNetV2 első rétegének 32 szűrője")
plt.tight_layout()
plt.show()

### 9.2 Feature mapek különböző mélységekben

Egy tesztképet átengedünk a hálón, és megnézzük, mit ad ki négy különböző mélységű réteg. Minden rétegből a 8 legerősebben aktiválódó feature mapet mutatjuk (világos = erős aktiváció).

Figyeld meg, hogyan változik a mélységgel
- a feature mapek **mérete** (felbontása),
- a feature mapek **száma** (csatornák),
- és hogy mennyire **felismerhető** még rajtuk az eredeti kép.

In [ ]:
layer_names = ["Conv1_relu", "block_3_expand_relu", "block_6_expand_relu", "block_13_expand_relu"]
layer_titles = ["1. réteg (korai)", "block 3 (korai-középső)", "block 6 (középső)", "block 13 (késői)"]

fmap_model = keras.Model(inputs=base_model.inputs,
                         outputs=[base_model.get_layer(n).output for n in layer_names])

img = test_images[0].astype("float32")
fmaps = fmap_model(rescale(img[None]))

plt.figure(figsize=(3, 3))
plt.imshow(test_images[0])
plt.title(f"Bemeneti kép ({CLASS_NAMES[y_true[0]]})")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(len(layer_names), 8, figsize=(14, 7.5))
for row, (title, fmap) in enumerate(zip(layer_titles, fmaps)):
    fmap = np.array(fmap[0])
    h, w, c = fmap.shape
    strongest = np.argsort(fmap.mean(axis=(0, 1)))[::-1][:8]
    for j, ch in enumerate(strongest):
        ax = axes[row, j]
        ax.imshow(fmap[:, :, ch], cmap="viridis")
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"{title}\n{h}×{w}×{c}", fontsize=8)
plt.suptitle("Feature mapek különböző mélységekben (sorok: rétegek, oszlopok: csatornák)")
plt.tight_layout()
plt.show()

### 9.3 Grad-CAM: hová néz a modell?

A **Grad-CAM** (*Gradient-weighted Class Activation Mapping*) módszer megmutatja, hogy a kép mely részei járultak hozzá leginkább a modell döntéséhez. Röviden így működik:
1. kiszámoljuk, hogy a győztes osztály valószínűsége mennyire érzékeny az utolsó konvolúciós réteg egyes feature mapjeire (gradiens),
2. a feature mapeket ezekkel a súlyokkal összegezzük,
3. az eredményt (egy 5×5-ös térképet) felnagyítjuk a kép méretére, és rárajzoljuk a képre.

A piros területek voltak a legfontosabbak a döntésben. **Egy jó modell a virágra néz, nem a háttérre.**

In [ ]:
def grad_cam(img):
    x = rescale(tf.convert_to_tensor(img[None], dtype=tf.float32))
    with tf.GradientTape() as tape:
        conv_out = base_model(x, training=False)    # 1 × 5 × 5 × 1280
        tape.watch(conv_out)
        preds = classifier(gap(conv_out))
        top = int(tf.argmax(preds[0]))
        score = preds[0, top]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))           # feature mapenkénti fontosság
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    heatmap = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0]
    return heatmap.numpy(), top, float(score)


rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(test_images), 4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, idx in enumerate(sample_idx):
    heatmap, top, p = grad_cam(test_images[idx].astype("float32"))
    axes[0, col].imshow(test_images[idx])
    axes[0, col].set_title(f"valós: {CLASS_NAMES[y_true[idx]]}")
    axes[1, col].imshow(test_images[idx])
    axes[1, col].imshow(heatmap, cmap="jet", alpha=0.45)
    axes[1, col].set_title(f"tipp: {CLASS_NAMES[top]} ({p:.0%})")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
plt.suptitle("Grad-CAM: a döntés szempontjából fontos képrészletek (piros = fontos)")
plt.tight_layout()
plt.show()

## 10. Próbáld ki a saját képeddel!

Futtasd a cellát, majd a megjelenő **„Choose files”** gombbal tölts fel egy vagy több képet a gépedről (jpg vagy png). A modell kiírja, melyik virágnak tartja.

Próbálj ki egy **virágot nem ábrázoló** képet is (pl. egy macskát vagy egy autót), és figyeld meg, mit mond a modell!

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    paths = list(uploaded.keys())
except ImportError:
    print("A feltöltés csak Google Colabban működik. Más környezetben írd be kézzel a kép(ek) elérési útját:")
    paths = []   # pl. paths = ["sajat_virag.jpg"]

for path in paths:
    img = keras.utils.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras.utils.img_to_array(img)
    probs = tl_model.predict(arr[None], verbose=0)[0]
    order = np.argsort(probs)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.imshow(img)
    ax1.set_title(path, fontsize=9)
    ax1.axis("off")
    ax2.barh([CLASS_NAMES[i] for i in order], probs[order], color="tab:green")
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("valószínűség")
    ax2.set_title(f"Tipp: {CLASS_NAMES[order[-1]]} ({probs[order[-1]]:.0%})")
    plt.tight_layout()
    plt.show()

## 11. A modell mentése (opcionális)

A betanított modellt elmentheted, és később, például a következő feladatban, újra betöltheted a `keras.models.load_model("virag_mobilenetv2.keras")` hívással. A fájl a bal oldali **Files** (mappa ikon) panelen jelenik meg, onnan le is töltheted.

> A Colab-munkamenet lejárta után a mentett fájl törlődik, ezért ha meg szeretnéd tartani, töltsd le!

In [ ]:
tl_model.save("virag_mobilenetv2.keras")
print("Mentve: virag_mobilenetv2.keras")

## Összefoglalás

- Egy nulláról tanított CNN kevés adatból csak korlátozott pontosságot ér el, mert az általános képi jellemzőket (éleket, textúrákat, formákat) is ebből a pár száz képből kellene megtanulnia.
- Egy előtanított háló ezeket a jellemzőket már „tudja”, ezért **feature extractionnel** egyetlen új réteg tanításával is sokkal jobb eredményt kapunk, töredék tanítható paraméterrel.
- A **fine-tuning** kis tanulási rátával tovább javíthat, mert a felső rétegeket hozzáigazítja a saját feladatunkhoz.
- A háló rétegei hierarchikusan dolgoznak: az első réteg szűrői egyszerű éleket és színeket keresnek, a mélyebb rétegek egyre absztraktabb jellemzőket.

## 12. Kérdések (beadandó)

Válaszolj röviden (2–5 mondatban) az alábbi kérdésekre! A válaszokat a kérdések alatti **„Válasz”** cellákba írd: kattints duplán a cellára a szerkesztéshez, majd Shift + Enter a mentéshez.

**Beadás:** *File → Download → Download .ipynb*, majd töltsd fel a kapott fájlt.

**1. kérdés.** Hasonlítsd össze a három modell teszt pontosságát (7. fejezet táblázata)! Mekkora a különbség a nulláról tanított CNN és a transfer learning között, és mi ennek az oka?

**Válasz:** *ide írd a válaszodat*

**2. kérdés.** Hány tanítható paramétere volt a modellnek feature extraction, illetve fine-tuning közben? Miért tanítható a feature extraction ilyen gyorsan, és miért kevésbé hajlamos a túltanulásra?

**Válasz:** *ide írd a válaszodat*

**3. kérdés.** Miért használtunk a fine-tuningnál 100-szor kisebb tanulási rátát, mint a feature extractionnél? Miért nem a fine-tuninggal kezdtük?

**Válasz:** *ide írd a válaszodat*

**4. kérdés.** Mit ábrázolnak az első konvolúciós réteg szűrői (9.1)? Hogyan változik a rétegek mélységével a feature mapek mérete, száma és „olvashatósága” (9.2)? Miért ez teszi lehetővé a transfer learninget?

**Válasz:** *ide írd a válaszodat*

**5. kérdés.** A konfúziós mátrix alapján melyik két osztályt keveri össze leggyakrabban a modell? Nézd meg a hibás képeket is: mi lehet a hibák vizuális oka?

**Válasz:** *ide írd a válaszodat*

**6. kérdés.** A Grad-CAM ábrák alapján a virágra vagy a háttérre „figyel” a modell? Mutatott-e valamelyik kép meglepő eredményt?

**Válasz:** *ide írd a válaszodat*

**7. kérdés.** Mit mondott a modell egy nem virágot ábrázoló képre (10. fejezet)? Miért viselkedik így, és hogyan lehetne ezt a problémát kezelni?

**Válasz:** *ide írd a válaszodat*

**8. bónuszkérdés (nem kötelező).** A Beállítások cellában írd át a `TRAIN_SPLIT` értékét `"train[:5%]"`-ra (kb. 180 tanítókép), majd futtasd újra a teljes notebookot (*Runtime → Restart session and run all*). Hogyan változott a baseline és a transfer learning pontossága? Mit mutat ez a transfer learning előnyéről?

**Válasz:** *ide írd a válaszodat*